In [ ]:
# =============================================================================
# explore/Explore_live_readings.py
# Filter and plot the EA real-time readings, archive and live tail together.
#
# RUN ALL CELLS, then use the controls under the last cell:
#   - Narrow by EA area, river, parameter and station. Each list only offers choices that
#     still lead somewhere, so you cannot pick an empty combination.
#   - Choose one or more measures (up to 8; different units get their own panel).
#   - Drag the slider for how many days back to show (1 to 30).
#   - The dashed red line is where the archive ends and the live tail begins: midnight at
#     the end of the last day the daily archive has landed in bronze (job 10). Everything
#     to the right, in the shaded band, is live (job 23) and not yet archived.
#
# READ ONLY. It reads ea_rt_readings_all (the archive plus the live tail as one series),
# ea_rt_measure_register and ea_rt_readings_bronze. Nothing is written.
#
# By hand, not scheduled. Uses ipywidgets and matplotlib, both in the Databricks runtimes.
#
# Attribution (OGL), wherever these readings are shown:
#   "this uses Environment Agency flood and river level data from the
#    real-time data API (Beta)"
# =============================================================================

import os
import sys

# Resolve the project root from this notebook's own location, so nothing
# depends on one person's workspace folder.
_nb_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
if not _nb_path.startswith("/Workspace"):
    _nb_path = "/Workspace" + _nb_path
PROJECT_ROOT = os.path.dirname(os.path.dirname(_nb_path))
sys.path.insert(0, PROJECT_ROOT)

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import ipywidgets as widgets
from datetime import timezone
from IPython.display import display, clear_output

from config import TBL_EA_RT_ALL, TBL_EA_RT_BRONZE, TBL_EA_RT_REGISTER, TBL_EA_RT_LIVE
from utils.explore import (
    ALL, prepare_register, filter_register, options, archive_live_boundary,
    window_start, to_utc, summarise
)

MAX_MEASURES = 8      # more lines than this stops being readable
MAX_DAYS     = 30     # the archive holds at least this many days

# The register: one row per measure, with the station, river, area and what is measured.
register = prepare_register(spark.table(TBL_EA_RT_REGISTER).toPandas())
print(f"Register: {len(register):,} measures at {register['station'].nunique():,} stations.")


In [ ]:
# =============================================================================
# DATA: THE READINGS, AND WHERE THE ARCHIVE ENDS
# =============================================================================

def archive_boundary():
    """Midnight UTC at the end of the last archived day, or None if nothing is archived."""
    row = spark.sql(f"SELECT MAX(reading_date) AS d FROM {TBL_EA_RT_BRONZE}").collect()[0]
    return archive_live_boundary(row["d"])


def load_readings(measure_uris, days):
    """Readings for the chosen measures over the last `days` days, archive and live together."""
    uris = ", ".join("'" + u.replace("'", "''") + "'" for u in measure_uris)
    pdf = spark.sql(f"""
        SELECT measure_uri, date_time, value
        FROM {TBL_EA_RT_ALL}
        WHERE measure_uri IN ({uris})
          AND date_time >= current_timestamp() - INTERVAL {int(days)} DAYS
        ORDER BY measure_uri, date_time
    """).toPandas()
    pdf["date_time"] = to_utc(pdf["date_time"])
    return pdf


In [ ]:
# =============================================================================
# THE CHART
# One panel per unit (levels in metres and flows in cubic metres a second do not share
# an axis). The shaded band and the dashed red line mark the live tail.
# =============================================================================

def draw(readings, chosen, days, boundary):
    now   = pd.Timestamp.now(tz="UTC")
    start = window_start(now, days)
    names = chosen.set_index("measure_uri")
    units = sorted({str(names.loc[u, "unit_name"]) if pd.notna(names.loc[u, "unit_name"]) else "(no unit)"
                    for u in readings["measure_uri"].unique()}) or ["(no unit)"]

    fig, axes = plt.subplots(len(units), 1, sharex=True, figsize=(11.5, 3.4 * len(units)), squeeze=False)
    axes = axes[:, 0]
    unit_axis = dict(zip(units, axes))
    colours = plt.rcParams["axes.prop_cycle"].by_key()["color"]

    for k, (uri, grp) in enumerate(readings.groupby("measure_uri")):
        unit = str(names.loc[uri, "unit_name"]) if pd.notna(names.loc[uri, "unit_name"]) else "(no unit)"
        label = f"{names.loc[uri, 'station']}: {names.loc[uri, 'measure_name']}"
        unit_axis[unit].plot(grp["date_time"], grp["value"], lw=1.4, color=colours[k % len(colours)], label=label)

    for unit, ax in unit_axis.items():
        if boundary is not None and boundary < now:
            ax.axvspan(max(boundary, start), now, color="#fff1cc", alpha=0.8, zorder=0)
            if start < boundary:
                ax.axvline(boundary, color="#c0392b", ls="--", lw=1.8, zorder=3)
                ax.text(boundary, 1.0, " archive | live", transform=ax.get_xaxis_transform(),
                        color="#c0392b", fontsize=9, va="bottom", ha="left")
        ax.set_ylabel(unit)
        ax.grid(alpha=0.3)
        ax.legend(loc="upper left", fontsize=8, frameon=False)
    axes[-1].set_xlim(start, now)
    axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%a %d %b %H:%M", tz=timezone.utc))
    fig.autofmt_xdate()
    fig.suptitle(f"Last {days} day{'s' if days != 1 else ''}, times in UTC", fontsize=11, x=0.01, ha="left")
    fig.tight_layout()
    return fig, now


In [ ]:
# =============================================================================
# THE CONTROLS
# Narrow by area, river, parameter and station (each list narrows by the others), pick
# measures, drag the slider. The chart redraws when the measures or the slider change.
# =============================================================================

opts = options(register)
w_area    = widgets.Dropdown(options=opts["areas"],      value=ALL, description="EA area",  layout=widgets.Layout(width="360px"))
w_river   = widgets.Dropdown(options=opts["rivers"],     value=ALL, description="River",    layout=widgets.Layout(width="360px"))
w_param   = widgets.Dropdown(options=opts["parameters"], value=ALL, description="Parameter", layout=widgets.Layout(width="360px"))
w_station = widgets.Dropdown(options=opts["stations"],   value=ALL, description="Station",  layout=widgets.Layout(width="360px"))
w_measure = widgets.SelectMultiple(options=[], description="Measures", rows=8, layout=widgets.Layout(width="620px"))
w_days    = widgets.IntSlider(value=3, min=1, max=MAX_DAYS, step=1, description="Days", continuous_update=False,
                              layout=widgets.Layout(width="620px"))
w_refresh = widgets.Button(description="Refresh", icon="refresh", tooltip="Fetch the latest readings again")
out       = widgets.Output()
_busy     = {"on": False}      # stops a change made by the code from triggering more changes


def _current():
    return filter_register(register, w_area.value, w_river.value, w_param.value, w_station.value)


def _sync_lists(changed=None):
    """Re-narrow every list by the others, keep a choice if it is still offered, refill the measures."""
    _busy["on"] = True
    try:
        o = options(register, w_area.value, w_river.value, w_param.value)
        for widget, key in ((w_area, "areas"), (w_river, "rivers"), (w_param, "parameters"), (w_station, "stations")):
            keep = widget.value if widget.value in o[key] else ALL
            widget.options = o[key]
            widget.value = keep
        rows = _current()
        entries = [(f"{r.station}: {r.measure_name}", r.measure_uri) for r in rows.itertuples()]
        entries.sort(key=lambda e: e[0].lower())
        w_measure.options = entries[:500]          # a very wide filter: ask for narrowing, not a huge list
        # A single station: preselect up to four of its measures, so the chart appears at once.
        w_measure.value = tuple(uri for _, uri in entries[:4]) if w_station.value != ALL else ()
    finally:
        _busy["on"] = False


def _redraw(_=None):
    if _busy["on"]:
        return
    with out:
        clear_output(wait=True)
        chosen_uris = list(w_measure.value)
        if not chosen_uris:
            print("Choose one or more measures. Pick a station to start, or narrow by river or area.")
            return
        if len(chosen_uris) > MAX_MEASURES:
            print(f"{len(chosen_uris)} measures chosen. Pick {MAX_MEASURES} or fewer so the chart stays readable.")
            return
        chosen = register[register["measure_uri"].isin(chosen_uris)].drop_duplicates("measure_uri")
        try:
            readings = load_readings(chosen_uris, w_days.value)
            boundary = archive_boundary()
        except Exception as e:
            print(f"Could not read the readings: {e}")
            print("Has 23_ea_rt_live_readings run? It creates ea_rt_readings_all once the archive (08, 10) exists.")
            return
        if readings.empty:
            print("No readings for these measures in that window.")
            return
        fig, now = draw(readings, chosen, w_days.value, boundary)
        display(fig)
        plt.close(fig)
        s = summarise(readings, boundary, now)
        print(f"{s['readings']:,} readings: {s['archive']:,} archived, {s['live']:,} live (right of the red line).")
        print(f"Newest reading {s['newest']:%a %d %b %H:%M} UTC, {s['age_minutes']:g} minutes ago.")
        if boundary is None:
            print("Nothing is archived yet, so there is no archive | live line.")


for w in (w_area, w_river, w_param, w_station):
    w.observe(lambda change: (_sync_lists(), _redraw()) if not _busy["on"] and change["name"] == "value" else None, names="value")
w_measure.observe(lambda change: _redraw() if change["name"] == "value" else None, names="value")
w_days.observe(lambda change: _redraw() if change["name"] == "value" else None, names="value")
w_refresh.on_click(_redraw)

_sync_lists()
display(widgets.VBox([
    widgets.HBox([widgets.VBox([w_area, w_river, w_param, w_station]), widgets.VBox([w_measure, w_days, w_refresh])]),
    out,
]))
_redraw()   # shows the prompt, or the chart if a station is already chosen
